# Libra Tier 2 — Google Colab Quickstart (0-GPU Replay + 1-GPU Real Closed-Loop RL)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NetX-lab/Libra/blob/main/repro/tier2_four_gpu_closed_loop/colab_quickstart.ipynb)

This notebook runs two verification stages inside a standard Google Colab runtime (`Runtime -> Change runtime type -> T4 / L4 / A100 GPU`):

1. **Tier 1 (0-GPU CPU Replay)**: Runs Libra's heterogeneous schedulers (`round_robin`, `least_connections`, `length_aware`, `la_mlfq`, `cmlfq`, `cmlfq_cost`) and `GlobalResourcePlanner` in ~2 seconds.
2. **Tier 2 (1-GPU Co-Located Real Closed-Loop RL)**: Launches 1 real vLLM rollout worker (`InplaceReloadWorkerExtension`) and 1 `FSDPTrainEngine` DAPO-Math RL trainer sharing GPU 0 with confirmed in-place checkpoint reloads.

| Colab Runtime | VRAM | Auto Preset Model | Precision (`vLLM` / `FSDP`) | vLLM KV Fraction |
| :--- | :---: | :--- | :--- | :---: |
| **T4 (Free)** | 16 GB | `Qwen/Qwen2.5-0.5B-Instruct` | `half` / `float32` | `0.30` |
| **L4** | 24 GB | `Qwen/Qwen2.5-1.5B-Instruct` | `bfloat16` / `bfloat16` | `0.35` |
| **A100-40GB** | 40 GB | `Qwen/Qwen3-1.7B` | `bfloat16` / `bfloat16` | `0.38` |
| **A100-80GB** | 80 GB | `Qwen/Qwen3-4B` | `bfloat16` / `bfloat16` | `0.38` |


## 1. Inspect Colab GPU & Install Libra Runtime Dependencies


In [ ]:
import os, subprocess, torch

!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"Detected GPU: {props.name} | VRAM={props.total_memory / 1024**3:.1f} GiB | SM={props.major}.{props.minor}")

if not os.path.exists("Libra"):
    !git clone --depth 1 https://github.com/NetX-lab/Libra.git
%cd Libra

!pip -q install --upgrade vllm transformers "datasets>=2.18.0,<3.0.0" pyyaml httpx fastapi sympy regex latex2sympy2 word2number
!pip -q install -e . --no-deps

## 2. Tier 1 — 0-GPU Scheduling Replay & GRP Offline Benchmark (~2s)


In [ ]:
!python repro/tier1_zero_gpu_replay/run_replay.py

## 3. Tier 2 — Real 1-GPU Closed-Loop DAPO-Math Training

`launch_4gpu.py --mode colab-1gpu` automatically picks the safe precision & model tier (`--colab-tier auto`), starts a supervised `vllm_hot_reload_api_server.py` worker on GPU 0, runs `benchmark_scheduler_train.py` (`FSDPTrainEngine` + `DAPOMathWorkflow`), and triggers verified in-place weight reloads.


In [ ]:
!python repro/tier2_four_gpu_closed_loop/launch_4gpu.py \
  --mode colab-1gpu \
  --colab-tier auto \
  --steps 4 \
  --sync-interval 2 \
  --output-dir ./logs/colab_tier2

## 4. Inspect Training Metrics, Weight Reload Acks, and Replay Collected Rollout Trace


In [ ]:
import glob, json
from pathlib import Path

summary_path = Path("./logs/colab_tier2/summary.json")
if summary_path.exists():
    print("=== summary.json ===")
    print(summary_path.read_text())

steps_files = sorted(glob.glob("./logs/colab_tier2/*/steps.jsonl"))
if steps_files:
    print(f"=== Step Records ({steps_files[0]}) ===")
    for line in Path(steps_files[0]).read_text().splitlines():
        row = json.loads(line)
        print(f"step={row["step"]} stats={row["stats"]}")

acks = sorted(glob.glob("./logs/colab_tier2/*/control/ack_*.json"))
print(f"Confirmed in-place weight reload acks: {len(acks)}")
for ack in acks:
    print(" ", ack)

history_files = sorted(glob.glob("./logs/colab_tier2/*/history/*.jsonl"))
if history_files:
    print("\n=== Replaying Colab Rollout History with Tier 1 ===")
    !python repro/tier1_zero_gpu_replay/run_replay.py --trace {history_files[0]} --tp-layout 1 --online-only --skip-grp